In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Klasický feedforward a řízení toku (dynamické obvody)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Klasický feedforward a řízení toku


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících závislostí.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Nová verze dynamických obvodů je nyní dostupná všem uživatelům na všech Backend zařízeních. Dynamické obvody teď můžeš spouštět v užitkovém měřítku. Více informací najdeš v [oznámení](/announcements/product-updates/2025-09-25-new-dynamic-circuits).

Dynamické obvody jsou mocné nástroje, pomocí nichž lze měřit Qubity uprostřed provádění kvantového Circuit a poté v rámci Circuit provádět operace klasické logiky na základě výsledků těchto průběžných měření. Tento postup se také nazývá _klasický feedforward_. Přestože je porozumění tomu, jak nejlépe využít dynamické obvody, teprve v počátcích, komunita kvantového výzkumu již identifikovala řadu případů použití, například:

* Efektivní příprava kvantových stavů, například [GHZ stav,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W-stav,](https://arxiv.org/abs/2403.07604) (další informace o W-stavu najdeš také v ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) a rozsáhlá třída [maticových produktových stavů](https://arxiv.org/abs/2404.16083)
* [Efektivní provázání na velké vzdálenosti](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) mezi Qubity na stejném čipu pomocí mělkých obvodů
* Efektivní [vzorkování obvodů podobných IQP](https://arxiv.org/pdf/2505.04705)

Tato zlepšení přinášená dynamickými obvody jsou však spojena s kompromisy. Průběžná měření a klasické operace mají zpravidla delší dobu provádění než dvoQubitové Gate, a tento nárůst doby může negovat výhody snížené hloubky Circuit. Proto je zkrácení délky průběžného měření oblastí zlepšení, na níž IBM Quantum&reg; pracuje v rámci [nové verze](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) dynamických obvodů.

[Specifikace OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) definuje řadu struktur pro řízení toku, ale Qiskit Runtime v současnosti podporuje pouze podmíněný příkaz `if`. V Qiskit SDK toto odpovídá metodě [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) na [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Tato metoda vrací [správce kontextu](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) a typicky se používá v příkazu `with`. Tento průvodce popisuje, jak tento podmíněný příkaz používat.

> **Note:** Příklady kódu v tomto průvodci používají standardní instrukci measure pro průběžná měření. Doporučuje se však místo toho použít instrukci [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit), pokud ji Backend podporuje. Podrobnosti najdeš v [dokumentaci průběžných měření](/guides/measure-qubits#mid-circuit-measurements).
## Příkaz `if`
Příkaz `if` se používá k podmíněnému provádění operací na základě hodnoty klasického bitu nebo registru.

V níže uvedeném příkladu aplikujeme Hadamardovu Gate na Qubit a změříme jej. Pokud je výsledek 1, aplikujeme na Qubit Gate X, čímž jej přepneme zpět do stavu 0. Poté Qubit znovu změříme. Výsledný výsledek měření by měl být 0 se 100% pravděpodobností.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Příkazu `with` lze přiřadit cíl, který je sám správcem kontextu a lze jej uložit a následně použít k vytvoření bloku else, jenž se provede vždy, když se obsah bloku `if` *neprovede*.

V níže uvedeném příkladu inicializujeme registry se dvěma Qubity a dvěma klasickými bity. Na první Qubit aplikujeme Hadamardovu Gate a změříme jej. Pokud je výsledek 1, aplikujeme Hadamardovu Gate na druhý Qubit; v opačném případě aplikujeme Gate X na druhý Qubit. Nakonec změříme i druhý Qubit.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Kromě podmíněnosti na jednom klasickém bitu je také možné podmíněnost vztáhnout na hodnotu klasického registru složeného z více bitů.

V níže uvedeném příkladu aplikujeme Hadamardovy Gate na dva Qubity a změříme je. Pokud je výsledek `01`, tedy první Qubit je 1 a druhý Qubit je 0, aplikujeme Gate X na třetí Qubit. Nakonec změříme třetí Qubit. Všimni si, že pro přehlednost jsme se rozhodli specifikovat stav třetího klasického bitu, který je 0, v podmínce `if`. Ve schématu Circuit je podmínka znázorněna kruhy na klasických bitech, na které se podmínka vztahuje. Černý kruh označuje podmíněnost na 1, zatímco bílý kruh označuje podmíněnost na 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Klasické výrazy
Modul klasických výrazů Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) obsahuje experimentální reprezentaci runtime operací na klasických hodnotách během provádění Circuit. Kvůli hardwarovým omezením jsou v současnosti podporovány pouze podmínky `QuantumCircuit.if_test()`.

Následující příklad ukazuje, že výpočet parity lze použít k vytvoření n-Qubitového GHZ stavu pomocí dynamických obvodů. Nejprve vygeneruj $n/2$ Bellových párů na sousedních Qubitech. Poté tyto páry slepuj dohromady pomocí vrstvy CNOT Gate mezi páry. Následně změř cílový Qubit všech předchozích CNOT Gate a každý změřený Qubit resetuj do stavu $\vert 0 \rangle$. Aplikuj $X$ na každé neměřené místo, pro které je parita všech předchozích bitů lichá. Nakonec se na změřené Qubity aplikují CNOT Gate, aby se obnovilo provázání ztracené při měření.

Při výpočtu parity první prvek konstruovaného výrazu zahrnuje zvednutí Python objektu `mr[0]` na uzel [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` se používá k převodu libovolných objektů na klasické výrazy). To není nutné pro `mr[1]` a případný následující klasický registr, protože jsou vstupem do `expr.bit_xor` a veškeré potřebné zvednutí se v těchto případech provede automaticky. Takové výrazy lze sestavovat ve smyčkách a dalších konstruktech.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Vyhledání Backend zařízení podporujících dynamické obvody
Chceš-li najít všechna Backend zařízení, ke kterým má tvůj účet přístup a která podporují dynamické obvody, spusť kód podobný následujícímu. Tento příklad předpokládá, že máš [uloženy přihlašovací údaje.](/guides/save-credentials) Můžeš také [explicitně zadat přihlašovací údaje](/guides/initialize-account#explicit) při inicializaci účtu Qiskit Runtime service. To ti například umožní zobrazit Backend zařízení dostupná pro konkrétní instanci nebo typ plánu.

> **Note:** - Backend zařízení dostupná pro účet závisí na instanci zadané v přihlašovacích údajích.
> - Nová verze dynamických obvodů je nyní dostupná všem uživatelům na všech Backend zařízeních. Více informací najdeš v [oznámení](/announcements/product-updates/2025-09-25-new-dynamic-circuits).